# KYC - CRNN recogniser fine-tuning (Kaggle GPU -> ONNX)

Fine-tunes the Persian text and digit/date recognisers from EasyOCR's own
pretrained Arabic-script weights, instead of training from random init -
see training/kaggle/README.md for why. One run, start to finish.

**Before you run:**

1. Settings > Accelerator > **GPU T4 x2** (a P100 cannot run current torch -
   this notebook checks and falls back to CPU rather than hanging, but T4 is
   far faster)
2. Settings > Internet > **On** (needed once, to download EasyOCR's
   pretrained weights - no API key required, it is a public release)
3. Add Data > attach the `crnn-bootstrap` dataset
   (`python -m scripts.kaggle_run upload-data --path .data/crnn_bootstrap
   --slug crnn-bootstrap --confirm` from the repo)

The exported `.onnx` + `.charset.txt` pairs land in `/kaggle/working/models/`.
Fetch them with `python -m scripts.kaggle_run pull`, or download from the
Output tab.


## 1. Configuration - the only cell you edit

In [ ]:
# Kaggle mounts a dataset under /kaggle/input/datasets/<owner>/<slug>, but that
# layout has not always held (see find_bootstrap_dir below) - this is only the
# first guess.
BOOTSTRAP_DIR_GUESS = "/kaggle/input/datasets/zhalehmanbari/crnn-bootstrap"

# Corrected label files, relative to the bootstrap dataset root. The "_combined"
# files are round 1 + round 2 of manual correction concatenated; see
# scripts/sample_bootstrap_labels.py for how new rounds get drawn without
# overlapping already-corrected rows.
TEXT_LABELS = "sample_text_combined.tsv"
DIGIT_LABELS = "sample_digits_combined.tsv"
DIGIT_CHARSET = "0123456789/"

TEXT_NAME = "crnn_fa_text"
DIGIT_NAME = "crnn_fa_digits"

EPOCHS = 80
BATCH = 16

# What to do when the assigned GPU is one this torch cannot drive (see the
# field-detector notebook - Kaggle has handed out Tesla P100s, sm_60, which
# current torch does not support).
#   "cpu"  - train on CPU anyway.
#   "fail" - stop, so you can switch the accelerator to T4 and re-run.
ON_UNUSABLE_GPU = "cpu"

# Source of the kyc package. scripts/kaggle_run.py uploads it as a private
# dataset and attaches it here.
SRC_DATASET = "/kaggle/input/kyc-src"

# Falling back to a git clone is opt-in. It runs whatever is on the repo's
# default branch, which is not necessarily the code you just pushed.
ALLOW_GIT_CLONE = False
REPO_URL        = "https://github.com/zhaleh197/kyc.git"


## 2. Dependencies

`--no-deps` on easyocr is load-bearing, the same way it was for ultralytics in
the field-detector notebook: installing easyocr normally pulls its own torch
pin, and Kaggle's preinstalled torch is the one thing that must not change -
recent torch wheels have dropped sm_60, which is what a Tesla P100 is. Every
other easyocr dependency (opencv, scikit-image, shapely, ...) is unrelated to
torch and safe to install normally.


In [ ]:
!pip install -q --no-deps easyocr
!pip install -q ninja pyclipper python-bidi scikit-image shapely pydantic-settings onnxruntime

import torch

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())

# DEVICE is passed to train_crnn.py's --device so its own selection never has
# to guess - see the long comment on this exact problem in that script.
DEVICE = "cpu"
if torch.cuda.is_available():
    major, minor = torch.cuda.get_device_capability(0)
    supported = torch.cuda.get_arch_list()
    print(f"gpu: {torch.cuda.get_device_name(0)}  sm_{major}{minor}")
    print("torch was built for:", " ".join(supported))

    if f"sm_{major}{minor}" in supported:
        DEVICE = "cuda"
    elif ON_UNUSABLE_GPU == "cpu":
        print(
            f"\nWARNING: this torch cannot drive sm_{major}{minor}. Falling back to CPU.\n"
            "         For a fast run: Settings > Accelerator > GPU T4 x2, then re-run."
        )
    else:
        raise SystemExit(
            f"Assigned GPU is sm_{major}{minor}, which this torch cannot drive.\n"
            "Switch Settings > Accelerator to GPU T4 x2 and re-run, or set "
            'ON_UNUSABLE_GPU = "cpu" to train on CPU anyway.'
        )
else:
    print("no GPU visible - training on CPU")

print("device:", DEVICE)


## 3. Get the source

Same discovery-by-marker-file approach as the field-detector notebook: Kaggle's
mount layout for an attached dataset has not reliably matched
`/kaggle/input/<slug>` in practice, so this searches for a marker file rather
than trusting a hard-coded path.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

WORK = Path("/kaggle/working/src")
INPUT = Path("/kaggle/input")


def find_source_tree(root):
    if not root.exists():
        return None
    for marker in root.rglob("scripts/roboflow_pull.py"):
        candidate = marker.parent.parent
        if (candidate / "kyc" / "__init__.py").exists():
            return candidate
    return None


print("attached under /kaggle/input:")
for entry in sorted(INPUT.rglob("*")) if INPUT.exists() else []:
    if entry.is_dir() and len(entry.relative_to(INPUT).parts) <= 2:
        print("   ", entry.relative_to(INPUT))

source_tree = Path(SRC_DATASET) if Path(SRC_DATASET).exists() else find_source_tree(INPUT)

if source_tree is not None:
    print("\nusing source tree:", source_tree)
    if WORK.exists():
        shutil.rmtree(WORK)
    shutil.copytree(source_tree, WORK)
elif ALLOW_GIT_CLONE:
    print("\nsource dataset missing; cloning", REPO_URL)
    if not WORK.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(WORK)], check=True)
else:
    raise SystemExit(
        f"No source tree found under /kaggle/input - looked for {SRC_DATASET} and "
        "for any directory containing scripts/roboflow_pull.py.\n"
        "Push it with:  python -m scripts.kaggle_run push --kernel-slug kyc-crnn-finetune "
        "--notebook training/kaggle/train_crnn_finetune.ipynb"
    )

os.chdir(WORK)
if str(WORK) not in sys.path:
    sys.path.insert(0, str(WORK))
print("cwd:", os.getcwd())


PROGRESS = Path("/kaggle/working/progress")
PROGRESS.mkdir(parents=True, exist_ok=True)


def mark(step, detail=""):
    path = PROGRESS / f"{step}.txt"
    path.write_text(str(detail), encoding="utf-8")
    print(f"[progress] {step} {detail}".rstrip(), flush=True)


def run(*args):
    """Run a step and stream its output - see the field-detector notebook for
    why (a bare subprocess.run hides the child's own diagnostics)."""
    label = args[0]
    print(">", " ".join(str(a) for a in args), flush=True)
    process = subprocess.Popen(
        [sys.executable, *args],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in process.stdout:
        print(line, end="", flush=True)
    code = process.wait()
    if code != 0:
        raise SystemExit(f"{label} exited with code {code} - see the output above")
    mark(f"ok-{Path(label).stem}")


mark("01-source-ready", os.getcwd())


## 4. Find the bootstrap dataset and the pretrained weights

The corrected labels + crops come from the attached `crnn-bootstrap` dataset.
The pretrained weights come from EasyOCR itself - letting it download them
(a public GitHub release, no key needed) is simpler and more robust than
uploading a ~50MB checkpoint by hand, and it is exactly how the weights this
project's own numbers are checked against were obtained locally.


In [ ]:
from pathlib import Path


def find_bootstrap_dir(root):
    """Same lesson as the source tree and the card dataset before it: do not
    trust a specific /kaggle/input layout, search for the dataset's shape."""
    guess = Path(BOOTSTRAP_DIR_GUESS)
    if (guess / TEXT_LABELS).exists():
        return guess
    for candidate in root.rglob(TEXT_LABELS):
        if (candidate.parent / DIGIT_LABELS).exists():
            return candidate.parent
    return None


bootstrap_dir = find_bootstrap_dir(Path("/kaggle/input"))
if bootstrap_dir is None:
    raise SystemExit(
        f"No dataset with {TEXT_LABELS} found under /kaggle/input. Attach "
        "crnn-bootstrap under Add Data, or upload it with "
        "`python -m scripts.kaggle_run upload-data --path .data/crnn_bootstrap "
        "--slug crnn-bootstrap --confirm`."
    )
print("using bootstrap dataset at", bootstrap_dir)
mark("02-bootstrap-found", str(bootstrap_dir))

import easyocr  # noqa: E402

# detector=False: this only needs the recognition weights, not the (separate,
# larger) CRAFT text-detection model - downloading it too would be wasted
# bandwidth and time for a project that already has its own field detector.
_ = easyocr.Reader(["fa"], gpu=False, detector=False, recognizer=True, verbose=False)

pretrained_candidates = list(Path.home().rglob("arabic.pth"))
if not pretrained_candidates:
    raise SystemExit("EasyOCR did not leave arabic.pth anywhere under the home directory after Reader() ran.")
pretrained_path = pretrained_candidates[0]
print("pretrained weights at", pretrained_path)
mark("03-pretrained-ready", str(pretrained_path))


## 5. Fine-tune and export both recognisers

In [ ]:
run("training/kaggle/train_crnn.py",
    "--labels", str(bootstrap_dir / TEXT_LABELS),
    "--name", TEXT_NAME,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--finetune-from", str(pretrained_path),
    "--out", "/kaggle/working/models")
mark("04-text-trained")

run("training/kaggle/train_crnn.py",
    "--labels", str(bootstrap_dir / DIGIT_LABELS),
    "--name", DIGIT_NAME,
    "--charset", DIGIT_CHARSET,
    "--epochs", str(EPOCHS),
    "--batch", str(BATCH),
    "--device", DEVICE,
    "--finetune-from", str(pretrained_path),
    "--out", "/kaggle/working/models")
mark("05-digits-trained")


## 6. Honest evaluation against the held-out test split

`eval_text.tsv` / `eval_digits.tsv` are the dataset's `test_` split - disjoint
from every image used in training, hand-corrected in full (see
scripts/sample_bootstrap_labels.py). Running the actual ONNX inference path
here, not the training script's internal validation split, is what makes this
number trustworthy rather than optimistic.


In [ ]:
import cv2

from kyc.core.config import get_settings
from kyc.modules.ocr.recognizers.crnn_onnx import CrnnOnnxRecognizer


def levenshtein(a, b):
    if a == b:
        return 0
    previous = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        current = [i]
        for j, cb in enumerate(b, 1):
            current.append(min(previous[j] + 1, current[j - 1] + 1, previous[j - 1] + (ca != cb)))
        previous = current
    return previous[-1]


def evaluate_eval_set(bucket, model_file, charset_file, allowlist=None):
    settings = get_settings()
    settings.models_dir = Path("/kaggle/working/models")
    recognizer = CrnnOnnxRecognizer.load(
        settings.model_path(model_file), settings.model_path(charset_file), ["CPUExecutionProvider"], name=bucket
    )
    lines = (bootstrap_dir / f"eval_{bucket}.tsv").read_text(encoding="utf-8").splitlines()
    exact = total = 0
    edit_sum = 0.0
    for line in lines:
        if not line.strip():
            continue
        rel, _, truth = line.partition("\t")
        image = cv2.imread(str(bootstrap_dir / rel))
        if image is None:
            continue
        pred = recognizer.recognize(image, allowlist=allowlist).text
        total += 1
        exact += int(pred == truth)
        edit_sum += levenshtein(pred, truth) / max(1, len(truth))
    print(f"[{bucket}] exact-match {exact}/{total} = {exact / max(1, total):.1%}   "
          f"edit-distance {edit_sum / max(1, total):.4f}")


evaluate_eval_set("text", f"{TEXT_NAME}.onnx", f"{TEXT_NAME}.charset.txt")
evaluate_eval_set("digits", f"{DIGIT_NAME}.onnx", f"{DIGIT_NAME}.charset.txt", allowlist=DIGIT_CHARSET)
mark("06-evaluated")


## 7. Verify the artifacts

In [ ]:
import hashlib
from pathlib import Path

out = Path("/kaggle/working/models")
for path in sorted(out.glob("*")):
    if path.suffix == ".pt":
        continue  # the .best.pt checkpoint stays local to the run, not shipped
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    print(f"{path.name:<28} {path.stat().st_size / 1e6:7.2f} MB  sha256:{digest}")

print()
print("Record the sha256, dataset and metrics in models/manifest.yaml.")
print("Fetch with:  python -m scripts.kaggle_run pull --kernel zhalehmanbari/kyc-crnn-finetune")
